# 異なるトークナイザの分割結果の比較

同じ日本語・英語のテキストを4つのトークナイザで分割し、分割結果を表で比較してみましょう。

- **MeCab**（`fugashi` + `unidic-lite`）：辞書ベースの日本語単語分割器
- **GPT-2**（`gpt2`）：英語中心に学習されたバイト単位 BPE
- **日本語 BERT**（`cl-tohoku/bert-base-japanese-v3`）：MeCab + WordPiece による日本語向けサブワード


サブワードは、単語より短い部分も扱える分割単位です。BPE（Byte Pair Encoding）は、頻繁に隣り合う単位を繰り返しまとめて語彙を作る方法です。

In [1]:
!pip install -q transformers  # トークナイザを統一的なインターフェイスで読み込む
!pip install -q fugashi  # 日本語単語分割器 MeCab を Python から呼び出すために使う
!pip install -q unidic-lite  # MeCab が使う軽量な日本語辞書

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.4/691.4 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 17.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 比較に使うテキスト

日本語と英語の短い文を1つずつ用意します。同じ文をすべてのトークナイザに与えて、分割のされ方を見比べます。

In [2]:
japanese_text = "自然言語処理を学ぼう。"
english_text = "I'm learning natural language processing!"

print("日本語:", japanese_text, f"（{len(japanese_text)} 文字）")
print("英語  :", english_text, f"（{len(english_text)} 文字）")

日本語: 自然言語処理を学ぼう。 （11 文字）
英語  : I'm learning natural language processing! （41 文字）


どちらも短い一文です。日本語は 11 文字、英語はその数倍の長さがあります。この「文字数」と、これから各トークナイザが出す「トークン数」がどれだけ食い違うかに注目していきましょう。

## 単語単位のトークン化：MeCab

日本語には単語間の空白がないため、どこが単語の境界かは自明ではありません。そこで辞書をもとに分割候補を並べ、最もそれらしい分割を選ぶ**単語分割器**が使われます。代表的なのが MeCab で、ここでは Python から手軽に呼べる `fugashi`（軽量辞書 `unidic-lite` 付き）を通して使います。

まず MeCab を「テキストを渡すと単語のリストが返る関数」として用意します。

In [3]:
import fugashi

mecab = fugashi.Tagger()  # unidic-lite 辞書を使う日本語単語分割器


def split_with_mecab(text: str) -> list[str]:
    """日本語テキストを MeCab で単語に分割し、表層形のリストを返す。"""
    return [word.surface for word in mecab(text)]


split_with_mecab(japanese_text)

['自然', '言語', '処理', 'を', '学ぼう', '。']

人間の直感に近い、読みやすい分割です。
分割の粒度は辞書によって変わります。ここでは `unidic-lite` を使っています。

## サブワードのトークナイザを読み込む

現代の LLM では、コーパスから統計的に語彙を獲得する**サブワード**トークン化が主流です。ここでは Hugging Face の `transformers` を使い、性格の異なる2つのトークナイザを読み込みます。`AutoTokenizer` を使えば、どれも同じインターフェイスで扱えます。

| モデル | 特徴 |
|---|---|
| `bert-base-multilingual-cased` | 多言語向けのサブワード |
| `cl-tohoku/bert-base-japanese-v3` | 日本語向けのサブワード（MeCab で単語に区切ってからサブワード分割） |

いずれもモデル名を指定するだけで読み込めます。

In [12]:
from transformers import AutoTokenizer


mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
ja_bert_tokenizer = AutoTokenizer.from_pretrained("cl-tohoku/bert-base-japanese-v3")

print(f"bert-base-multilingual-cased: 語彙サイズ = {mbert_tokenizer.vocab_size:,}")
print(f"cl-tohoku/bert-base-japanese-v3: 語彙サイズ = {ja_bert_tokenizer.vocab_size:,}")

bert-base-multilingual-cased: 語彙サイズ = 119,547
cl-tohoku/bert-base-japanese-v3: 語彙サイズ = 32,768


### サブワードの分割処理をそろえる

`transformers` のトークナイザでは、テキストを ID の列にし、その ID をトークン文字列に戻すと分割を確認できます。3つのトークナイザで繰り返すこの処理を `split_with_hf` にまとめます。

In [16]:
from transformers import PreTrainedTokenizerBase


def split_with_hf(tokenizer: PreTrainedTokenizerBase, text: str) -> list[str]:
    """テキストを分割し、トークン文字列のリストを返す。"""
    ids = tokenizer.encode(text, add_special_tokens=False)
    return tokenizer.convert_ids_to_tokens(ids)

print(split_with_hf(mbert_tokenizer, english_text))
print(split_with_hf(ja_bert_tokenizer, japanese_text))

['I', "'", 'm', 'learning', 'natural', 'language', 'processing', '!']
['自然', '言語', '処理', 'を', '学', '##ぼう', '。']


`cl-tohoku/bert-base-japanese-v3` の "##" は単語内部の分割であることを表します。
このトークナイザは、テキストをまず単語分割したのち、それぞれの単語をサブワード分割するという手順を踏んでいます。

## 分割結果を見比べる

同じテキストを MeCab とサブワードトークナイザで分割して比較しましょう。

In [18]:
print("=== 日本語テキストの分割結果 ===")
print("元の文:   ", japanese_text)
print("MeCab:     ", split_with_mecab(japanese_text))
print("多言語BERT:", split_with_hf(mbert_tokenizer, japanese_text))
print("日本語BERT:", split_with_hf(ja_bert_tokenizer, japanese_text))

print("\n=== 英語テキストの分割結果 ===")
print("元の文:   ", english_text)
print("MeCab:     ", split_with_mecab(english_text))
print("多言語BERT:", split_with_hf(mbert_tokenizer, english_text))
print("日本語BERT:", split_with_hf(ja_bert_tokenizer, english_text))

=== 日本語テキストの分割結果 ===
元の文:    自然言語処理を学ぼう。
MeCab:      ['自然', '言語', '処理', 'を', '学ぼう', '。']
多言語BERT: ['自', '然', '言', '語', '処', '理', 'を', '学', 'ぼ', '##う', '。']
日本語BERT: ['自然', '言語', '処理', 'を', '学', '##ぼう', '。']

=== 英語テキストの分割結果 ===
元の文:    I'm learning natural language processing!
MeCab:      ['I', "'", 'm', 'learning', 'natural', 'language', 'processing', '!']
多言語BERT: ['I', "'", 'm', 'learning', 'natural', 'language', 'processing', '!']
日本語BERT: ['I', "'", 'm', 'le', '##ar', '##ning', 'n', '##at', '##ural', 'la', '##ng', '##u', '##age', 'pro', '##cess', '##ing', '!']


**MeCab (単語レベル)**
   * 日本語の「単語」を自然に切り分けます。英語も空白や記号をベースに単語分割できます。

**日本語 BERT**
   * 日本語用に構築されており、単語分割の後、「学ぼう」をさらにサブワードに分割しています。
   * 一方で、英語のテキストは細切れになりやすいです。

**多言語 BERT**
   * こちらは逆に、日本語が細切れになりやすいです。


## まとめ

同じテキストでも、トークナイザによって分割が変わります。
テキストのトークン数は、後続のモデルの処理効率にも大きく関わるため、目的に応じてトークナイザを設計することも大事になります。